In [ ]:
### here in this code we are preprocessing the leakfree data and bilding the corpus from regulatory documents

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# Import libraries

# Data handling
from datasets import Dataset
from pandas import to_pickle

# Other utils
import json
from re import compile
import os

In [ ]:

## 1.Loading Splitted leak free data


import json

# Training dataset
with open('/content/drive/MyDrive/Colab Notebooks/Obliqa_new/train.json') as f:
    data_train = json.load(f)

# Evaluation dataset
with open('/content/drive/MyDrive/Colab Notebooks/Obliqa_new/eval.json') as f:
    data_eval = json.load(f)

# Testing dataset
with open('/content/drive/MyDrive/Colab Notebooks/Obliqa_new/test.json') as f:
    data_test = json.load(f)

In [ ]:
len(data_train), len(data_eval), len(data_test)

(20573, 3191, 3116)

In [ ]:
## 2.basic cleaning and prerocessing of dataset and coverted into flattened Hugging face format

import json
from datasets import Dataset

def simple_cleaning(text: str) -> str:
    """Basic cleanup of text."""
    return text.replace("\n", " ").strip()

def build_dataset(split_file, save_path):
    """Flatten ObliQA split into (question, passage) pairs and save."""
    with open(split_file) as f:
        data = json.load(f)

    dataset_list = []
    for q in data:
        q_text = simple_cleaning(q.get("question") or q.get("Question") or "")
        q_id   = q.get("QuestionID", "")

        for p in q["Passages"]:
            dataset_list.append({
                "question_id": q_id,
                "question": q_text,
                "passage_id": p["PassageID"],
                "document_id": p["DocumentID"],
                "passage": simple_cleaning(p["Passage"])
            })

    dataset = Dataset.from_list(dataset_list)
    dataset.save_to_disk(save_path)
    print(f"✅ Saved {save_path} | Examples: {len(dataset)}")
    return dataset

=
# Build and Save Train/Eval/Test

base_dir = "/content/drive/MyDrive/Colab Notebooks/Obliqa_new"

train_dataset = build_dataset(f"{base_dir}/train.json", f"{base_dir}/train_dataset_clean")
eval_dataset  = build_dataset(f"{base_dir}/eval.json",  f"{base_dir}/eval_dataset_clean")
test_dataset  = build_dataset(f"{base_dir}/test.json",  f"{base_dir}/test_dataset_clean")


Saving the dataset (0/1 shards):   0%|          | 0/26883 [00:00<?, ? examples/s]

✅ Saved /content/drive/MyDrive/Colab Notebooks/Obliqa_new/train_dataset_clean | Examples: 26883


Saving the dataset (0/1 shards):   0%|          | 0/3733 [00:00<?, ? examples/s]

✅ Saved /content/drive/MyDrive/Colab Notebooks/Obliqa_new/eval_dataset_clean | Examples: 3733


Saving the dataset (0/1 shards):   0%|          | 0/3491 [00:00<?, ? examples/s]

✅ Saved /content/drive/MyDrive/Colab Notebooks/Obliqa_new/test_dataset_clean | Examples: 3491


In [ ]:

## 3.building the Corpus of documents from the regulatory documents for retriever to search and find the passages.
# Structured regulatory documents are available in the github page.

import os, json, pickle

def simple_cleaning(text: str) -> str:
    """Basic cleanup of passages."""
    return text.replace("\n", " ").strip()

def to_pickle(obj, path):
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    print(f"✅ Saved corpus to {path}")

# --------------------------
# Step 1: Load documents and build collection
# --------------------------
import os, json, pickle

def to_pickle(obj, path):
    with open(path, "wb") as f:
        pickle.dump(obj, f)

ndocs = 40
collection = []

for i in range(1, ndocs + 1):
    with open(os.path.join("/content/drive/MyDrive/Colab Notebooks/StructuredRegulatoryDocuments", f"{i}.json")) as f:
        doc = json.load(f)
        for psg in doc:
            collection.append(
                dict(
                    text=psg["PassageID"] + " " + psg["Passage"],
                    ID=psg["ID"],
                    DocumentId=psg["DocumentID"],
                    PassageId=psg["PassageID"],
                )
            )


# --------------------------
# Step 2: Create corpus dictionary
# --------------------------
corpus = {
    f"{doc['DocumentId']}-{doc['PassageId']}": doc["text"]
    for doc in collection
}

print(f"✅ Corpus size: {len(corpus)}")

# --------------------------
# Step 3: Save to Drive
# --------------------------
save_path = "/content/drive/MyDrive/Colab Notebooks/Obliqa_new/corpus_clean.pkl"
to_pickle(corpus, save_path)


✅ Corpus size: 13705
